# Pipeline do Projeto 2 — Extração Estatística e ML de Grafos Clínicos

Equipe **CGGJL**: Caio Melloni, Gabriel Lorhan, George Henrique, João Vitor Gonçalves e Lucas Guarnieri.

Este notebook demonstra a execução e a inspeção dos resultados das **5 etapas** do Projeto 2:
1. **Etapa 1:** Termos compostos (C-value, PMI) e associações MeSH;
2. **Etapa 2:** Detecção de fronteiras de menções com CRF (BIO);
3. **Etapa 3:** Classificação de tipos clínicos (Léxico + Regressão Logística) e polaridade;
4. **Etapa 4:** Extração de relações clínicas com probabilidade/confiança;
5. **Etapa 5:** Montagem final do grafo de conhecimento e avaliação comparativa (P1 × P2).

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

# Localiza a raiz do repositório e o diretório do Projeto 2
cwd = Path.cwd()
ROOT = next(p for p in (cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent) if (p / "project2").is_dir())
P2_DIR = ROOT / "project2"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repositório Raiz:", ROOT)
print("Diretório Projeto 2:", P2_DIR)

Repositório Raiz: C:\Users\georg\Estudos\unicamp\PNL-CGGJL
Diretório Projeto 2: C:\Users\georg\Estudos\unicamp\PNL-CGGJL\project2


## 1. Etapa 1 — Termos Compostos e Associações MeSH

Nesta etapa, N-gramas (2 a 4 palavras) são filtrados por classes gramaticais (POS do HMM) e pontuados via C-value e PMI.
Também calculamos associações por PMI entre códigos MeSH coocorrentes nos casos.

In [2]:
termos_path = P2_DIR / "data" / "termos.csv"
if termos_path.exists():
    df_termos = pd.read_csv(termos_path)
    print(f"Total de termos compostos extraídos: {len(df_termos)}")
    display(df_termos.head(10))
else:
    print("Arquivo termos.csv ainda não encontrado.")

Total de termos compostos extraídos: 584
                     termo  n  frequencia       pmi    c_value
0     physical examination  2          20  8.719032  20.000000
1  spontaneous respiration  2          15  9.888957  15.000000
2                  s serum  2          13  6.840785  13.000000
3   white blood cell count  4           6  6.696664  12.000000
4       left main bronchus  3           7  7.753248  11.094738
5             normal range  2          11  8.345574  11.000000
6           abdominal pain  2          13  7.625056  11.000000
7      intensive care unit  3           6  9.719032   9.509775
8     past medical history  3           6  8.037208   9.509775
9        postoperative day  2           9  7.403799   9.000000


In [3]:
assoc_path = P2_DIR / "data" / "associacoes.csv"
if assoc_path.exists():
    df_assoc = pd.read_csv(assoc_path)
    print(f"Total de associações MeSH calculadas: {len(df_assoc)}")
    display(df_assoc.sort_values(by="pmi", ascending=False).head(10))
else:
    print("Arquivo associacoes.csv ainda não encontrado.")

Total de associações MeSH calculadas: 18
  codigo_a                     termo_a codigo_b  ... casos_a  casos_b       pmi
0  D000418                    Albumins  D004724  ...       2        5  3.263034
1  D005124           Eye Abnormalities  D008279  ...       4        4  3.169925
2  D006454                 Hemoglobins  D007958  ...       5        3  2.678072
3  D004322                    Drainage  D009369  ...       3        8  2.000000
4  D001706                      Biopsy  D008775  ...      10        3  1.678072
5  D003404                  Creatinine  D014054  ...       2       15  1.678072
6  D005124           Eye Abnormalities  D009369  ...       4        8  1.584963
7  D008279  Magnetic Resonance Imaging  D009369  ...       4        8  1.584963
8  D001706                      Biopsy  D006454  ...      10        5  1.526069
9  D008279  Magnetic Resonance Imaging  D014054  ...       4       15  1.263034

[10 rows x 8 columns]


## 2. Etapa 2 — Detecção de Fronteiras com CRF (BIO)

O modelo CRF linear (`sklearn-crfsuite`) prevê anotações BIO sem tipo sobre as sentenças dos casos clínicos.

In [4]:
mencoes_path = P2_DIR / "data" / "mencoes.csv"
if mencoes_path.exists():
    df_mencoes = pd.read_csv(mencoes_path)
    print(f"Total de menções previstas pelo CRF: {len(df_mencoes)}")
    display(df_mencoes.head(10))
else:
    print("Arquivo mencoes.csv não encontrado.")

Total de menções previstas pelo CRF: 754
          case_id   id  ...   end                                              texto
0  PMC10106591_01   m1  ...   222                                         chest pain
1  PMC10106591_01   m2  ...   408  tightness in her throat without shortness of b...
2  PMC10106591_01   m3  ...   422                                       palpitations
3  PMC10106591_01   m4  ...   438                                        diaphoresis
4  PMC10106591_01   m5  ...   761       abnormal glucose tolerance for the past year
5  PMC10106591_01   m6  ...  1023                            oxygen saturation level
6  PMC10106591_01   m7  ...  1030                                                99%
7  PMC10106591_01   m8  ...  1612         reduced left ventricular ejection fraction
8  PMC10106591_01   m9  ...  1622                                                50%
9  PMC10106591_01  m10  ...  1710       laboratory tests indicated normal troponin I

[10 rows x 5 columns]


## 3. Etapa 3 — Classificação de Tipos, Conceitos e Polaridade

Esta etapa opera em cascata:
1. Busca em dicionários e gazetteers MeSH / Anatomia;
2. Menções não resolvidas são classificadas por `LogisticRegression`;
3. A polaridade é detectada analisando a oração precedente.

In [5]:
tipadas_path = P2_DIR / "data" / "mencoes_tipadas.csv"
if tipadas_path.exists():
    df_tipadas = pd.read_csv(tipadas_path)
    print(f"Total de menções tipadas: {len(df_tipadas)}")
    print("\nDistribuição por Tipo de Entidade:")
    display(df_tipadas['tipo'].value_counts())
    print("\nDistribuição por Polaridade:")
    display(df_tipadas['polaridade'].value_counts())
    display(df_tipadas.head(10))
else:
    print("Arquivo mencoes_tipadas.csv não encontrado.")

Total de menções tipadas: 754

Distribuição por Tipo de Entidade:
tipo
Exam              199
ExamResult        164
Outcome            99
Diagnosis          89
Medication         60
AnatomicalSite     50
Finding            44
Symptom            30
Treatment          18
History             1
Name: count, dtype: int64

Distribuição por Polaridade:
polaridade
present    702
absent      52
Name: count, dtype: int64
          case_id   id  start  ...  polaridade confianca          fonte
0  PMC10106591_01   m1    212  ...     present    1.0000         lexico
1  PMC10106591_01   m2    357  ...     present    1.0000         lexico
2  PMC10106591_01   m3    410  ...      absent    0.3909  classificador
3  PMC10106591_01   m4    427  ...      absent    0.3682  classificador
4  PMC10106591_01   m5    717  ...     present    1.0000         lexico
5  PMC10106591_01   m6   1000  ...     present    1.0000         lexico
6  PMC10106591_01   m7   1027  ...     present    0.3734  classificador
7  PMC1010

## 4. Etapa 4 — Relações Clínicas com Confiança

As relações de paciente são associadas pelo tipo da entidade, enquanto relações entre menções são preditas por `LogisticRegression` baseada em N-gramas textuais e restrições da ontologia.

In [6]:
relacoes_path = P2_DIR / "data" / "relacoes.csv"
if relacoes_path.exists():
    df_rel = pd.read_csv(relacoes_path)
    print(f"Total de relações previstas: {len(df_rel)}")
    print("\nDistribuição por Tipo de Relação:")
    display(df_rel['relacao'].value_counts())
    display(df_rel.head(10))
else:
    print("Arquivo relacoes.csv não encontrado.")

Total de relações previstas: 718

Distribuição por Tipo de Relação:
relacao
UNDERWENT_EXAM    199
HAS_RESULT        123
HAS_OUTCOME        99
DIAGNOSED_WITH     89
TREATED_WITH       78
LOCATED_IN         39
HAS_SYMPTOM        30
REVEALS            27
HAS_FINDING        17
SUPPORTS           16
HAS_HISTORY         1
Name: count, dtype: int64
          case_id origem destino         relacao  confianca
0  PMC10106591_01     P1      m1     HAS_SYMPTOM        1.0
1  PMC10106591_01     P1      m2  DIAGNOSED_WITH        1.0
2  PMC10106591_01     P1      m3     HAS_SYMPTOM        1.0
3  PMC10106591_01     P1      m4     HAS_SYMPTOM        1.0
4  PMC10106591_01     P1      m5    TREATED_WITH        1.0
5  PMC10106591_01     P1      m6    TREATED_WITH        1.0
6  PMC10106591_01     P1      m8  UNDERWENT_EXAM        1.0
7  PMC10106591_01     P1     m10    TREATED_WITH        1.0
8  PMC10106591_01     P1     m12    TREATED_WITH        1.0
9  PMC10106591_01     P1     m14    TREATED_WITH        

## 5. Etapa 5 — Montagem do Grafo e Avaliação

A montagem funde menções de mesma entidade, normaliza unidades de medida, ancora nós `Concept` MeSH com arestas `SAME_AS` e preserva evidências e graus de confiança.

Abaixo inspecionamos os nós e arestas exportados para o caso `PMC5137649_01`:

In [7]:
caso_demo = "PMC5137649_01"
nodes_path = P2_DIR / "data" / "grafo" / f"{caso_demo}-nodes.csv"
edges_path = P2_DIR / "data" / "grafo" / f"{caso_demo}-edges.csv"

if nodes_path.exists() and edges_path.exists():
    df_nodes = pd.read_csv(nodes_path)
    df_edges = pd.read_csv(edges_path)
    print(f"Caso {caso_demo}: {len(df_nodes)} nós e {len(df_edges)} arestas.")
    print("\nNós do Grafo:")
    display(df_nodes.head(8))
    print("\nArestas do Grafo com Confiança:")
    display(df_edges.head(8))
else:
    print(f"Arquivos de grafo para {caso_demo} não encontrados.")

Caso PMC5137649_01: 44 nós e 55 arestas.

Nós do Grafo:
         case_id  ...                                         attributes
0  PMC5137649_01  ...                                                NaN
1  PMC5137649_01  ...  polarity=present; mentions=m1; mention_texts=[...
2  PMC5137649_01  ...  polarity=present; mentions=m2; mention_texts=[...
3  PMC5137649_01  ...  polarity=present; mentions=m3; mention_texts=[...
4  PMC5137649_01  ...  polarity=present; mentions=m4; mention_texts=[...
5  PMC5137649_01  ...  polarity=present; mentions=m5; mention_texts=[...
6  PMC5137649_01  ...  polarity=present; mentions=m6|m20; mention_tex...
7  PMC5137649_01  ...  polarity=present; mentions=m7|m10|m30|m31; men...

[8 rows x 5 columns]

Arestas do Grafo com Confiança:
         case_id  ...                                         attributes
0  PMC5137649_01  ...  mention_pairs=[["P1","m1"]]; evidence_spans=[[...
1  PMC5137649_01  ...  mention_pairs=[["P1","m2"]]; evidence_spans=[[...
2  PMC5137649

## Confronto P1 × P2 e Resultados Oficiais

Métricas finais avaliadas contra os 8 casos anotados no gabarito manual:

In [8]:
comp_path = P2_DIR / "data" / "resultados" / "comparativo_p1_p2.csv"
if comp_path.exists():
    df_comp = pd.read_csv(comp_path)
    display(df_comp)
else:
    print("Arquivo comparativo_p1_p2.csv não encontrado.")

                                               etapa  ...                       p2_f1
0                    Etapa 2 - Fronteiras de Mencoes  ...     0.0952 (pos_bio=0.2051)
1                    Etapa 2 - Fronteiras de Mencoes  ...     0.2063 (pos_bio=0.4615)
2             Etapa 3 - Tipagem Semantica (Macro-F1)  ...                      0.6188
3             Etapa 3 - Tipagem Semantica (Macro-F1)  ...                      0.1907
4        Etapa 3 - Polaridade (Acuracia / F1 absent)  ...  acc=0.9509 / f1_abs=0.7317
5         Etapa 4 - Extracao de Relacoes (F1 Global)  ...                        0.75
6         Etapa 4 - Extracao de Relacoes (F1 Global)  ...                         0.2
7  Etapa 5 - Grafo Estrutural (% Nos c/ MeSH | Du...  ...              0.4053 | dup=3

[8 rows x 7 columns]


## Visualização Interativa

Para visualizar os grafos clínicos no visualizador web Cytoscape.js com realce de evidências no texto original:

```bash
python -m http.server 8000 --directory project2/data/visualizador
```

Acesse `http://localhost:8000` no navegador.